# SASRec Attention-Bias Time-Aware BPI2012 Colab Train (`anchor_ml20` baseline)

Colab notebook for testing the strongest time-aware candidate so far on top of the final baseline `anchor_ml20`.

Design:
- baseline reuse: `anchor_ml20`
- time source: `delta_start_seconds`
- causal pairwise gap attention bias
- 9-bucket scalar attention bias
- evaluate under both `NDCG@10` and `NDCG@5` model-selection criteria


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
TIMEAWARE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10'
TIMEAWARE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('TIMEAWARE_NDCG10_OUTPUT_DIR:', TIMEAWARE_NDCG10_OUTPUT_DIR)
print('TIMEAWARE_NDCG5_OUTPUT_DIR:', TIMEAWARE_NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
BASELINE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5
TIMEAWARE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10
TIMEAWARE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG10_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [8]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Experiment design

Fixed baseline setting:
- `anchor_ml20`
- `hidden_units=32, num_blocks=2, num_heads=1, maxlen=20, lr=0.001, dropout=0.2`
- seeds: `42`, `2024`, `7`

Time-aware design:
- `delta_start_seconds`
- causal pairwise gap attention bias
- 9-bucket scalar bias
- no additive time embedding in this notebook


## Check existing baseline runs


In [10]:
from pathlib import Path

baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]

for label, output_dir in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR)),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR)),
]:
    print('=' * 80)
    print(label)
    for run_name in baseline_runs:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10
anchor_ml20_s42 EXISTS
anchor_ml20_s2024 EXISTS
anchor_ml20_s7 EXISTS
Baseline NDCG@5
anchor_ml20_s42 EXISTS
anchor_ml20_s2024 EXISTS
anchor_ml20_s7 EXISTS


## Check planned new runs


In [11]:
planned_ndcg10 = [
    'attnbias_dstart_ml20_b9_s42',
    'attnbias_dstart_ml20_b9_s2024',
    'attnbias_dstart_ml20_b9_s7',
]
planned_ndcg5 = [
    'attnbias_dstart_ml20_b9_s42',
    'attnbias_dstart_ml20_b9_s2024',
    'attnbias_dstart_ml20_b9_s7',
]

for label, output_dir, run_names in [
    ('Attention-Bias NDCG@10', Path(TIMEAWARE_NDCG10_OUTPUT_DIR), planned_ndcg10),
    ('Attention-Bias NDCG@5', Path(TIMEAWARE_NDCG5_OUTPUT_DIR), planned_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Attention-Bias NDCG@10
attnbias_dstart_ml20_b9_s42 OK
attnbias_dstart_ml20_b9_s2024 OK
attnbias_dstart_ml20_b9_s7 OK
Attention-Bias NDCG@5
attnbias_dstart_ml20_b9_s42 OK
attnbias_dstart_ml20_b9_s2024 OK
attnbias_dstart_ml20_b9_s7 OK


## Train attention-bias runs for `NDCG@10`


### attnbias_dstart_ml20_b9_s42


In [12]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml20_b9_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10/attnbias_dstart_ml20_b9_s42
epoch=1, loss=0.6454
epoch=2, loss=0.3058
epoch=3, loss=0.2128
epoch=4, loss=0.1694
epoch=5, loss=0.1455
valid [full], NDCG@5: 0.6387, HR@5: 0.7449, NDCG@10: 0.7119, HR@10: 0.9805, MRR: 0.6352
valid [sampled], NDCG@5: 0.5574, HR@5: 0.5584, NDCG@10: 0.5635, HR@10: 0.5781, MRR: 0.5734
test [full], NDCG@5: 0.7418, HR@5: 0.8532, NDCG@10: 0.7842, HR@10: 0.9892, MRR: 0.7211
test [sampled], NDCG@5: 0.1339, HR@5: 0.2152, NDCG@10: 0.2214, HR@10: 0.4860, MRR: 0.1696
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-awa

### attnbias_dstart_ml20_b9_s2024


In [13]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml20_b9_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10/attnbias_dstart_ml20_b9_s2024
epoch=1, loss=0.6694
epoch=2, loss=0.3203
epoch=3, loss=0.2175
epoch=4, loss=0.1708
epoch=5, loss=0.1452
valid [full], NDCG@5: 0.6214, HR@5: 0.7297, NDCG@10: 0.6938, HR@10: 0.9642, MRR: 0.6177
valid [sampled], NDCG@5: 0.5274, HR@5: 0.5276, NDCG@10: 0.5337, HR@10: 0.5478, MRR: 0.5453
test [full], NDCG@5: 0.7258, HR@5: 0.8567, NDCG@10: 0.7688, HR@10: 0.9903, MRR: 0.7020
test [sampled], NDCG@5: 0.2071, HR@5: 0.2695, NDCG@10: 0.2550, HR@10: 0.4164, MRR: 0.2343
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-a

### attnbias_dstart_ml20_b9_s7


In [14]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml20_b9_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10/attnbias_dstart_ml20_b9_s7
epoch=1, loss=0.6862
epoch=2, loss=0.3289
epoch=3, loss=0.2388
epoch=4, loss=0.1808
epoch=5, loss=0.1496
valid [full], NDCG@5: 0.6602, HR@5: 0.7985, NDCG@10: 0.7132, HR@10: 0.9634, MRR: 0.6406
valid [sampled], NDCG@5: 0.5497, HR@5: 0.5510, NDCG@10: 0.5556, HR@10: 0.5700, MRR: 0.5674
test [full], NDCG@5: 0.6524, HR@5: 0.8945, NDCG@10: 0.6835, HR@10: 0.9940, MRR: 0.5848
test [sampled], NDCG@5: 0.1813, HR@5: 0.1834, NDCG@10: 0.2007, HR@10: 0.2471, MRR: 0.2284
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-awar

## Train attention-bias runs for `NDCG@5`


### attnbias_dstart_ml20_b9_s42


In [15]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml20_b9_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5/attnbias_dstart_ml20_b9_s42
epoch=1, loss=0.6454
epoch=2, loss=0.3058
epoch=3, loss=0.2128
epoch=4, loss=0.1694
epoch=5, loss=0.1455
valid [full], NDCG@5: 0.6387, HR@5: 0.7449, NDCG@10: 0.7119, HR@10: 0.9805, MRR: 0.6352
valid [sampled], NDCG@5: 0.5574, HR@5: 0.5584, NDCG@10: 0.5635, HR@10: 0.5781, MRR: 0.5734
test [full], NDCG@5: 0.7418, HR@5: 0.8532, NDCG@10: 0.7842, HR@10: 0.9892, MRR: 0.7211
test [sampled], NDCG@5: 0.1339, HR@5: 0.2152, NDCG@10: 0.2214, HR@10: 0.4860, MRR: 0.1696
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware

### attnbias_dstart_ml20_b9_s2024


In [16]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml20_b9_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5/attnbias_dstart_ml20_b9_s2024
epoch=1, loss=0.6694
epoch=2, loss=0.3203
epoch=3, loss=0.2175
epoch=4, loss=0.1708
epoch=5, loss=0.1452
valid [full], NDCG@5: 0.6214, HR@5: 0.7297, NDCG@10: 0.6938, HR@10: 0.9642, MRR: 0.6177
valid [sampled], NDCG@5: 0.5274, HR@5: 0.5276, NDCG@10: 0.5337, HR@10: 0.5478, MRR: 0.5453
test [full], NDCG@5: 0.7258, HR@5: 0.8567, NDCG@10: 0.7688, HR@10: 0.9903, MRR: 0.7020
test [sampled], NDCG@5: 0.2071, HR@5: 0.2695, NDCG@10: 0.2550, HR@10: 0.4164, MRR: 0.2343
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-awa

### attnbias_dstart_ml20_b9_s7


In [17]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml20_b9_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5/attnbias_dstart_ml20_b9_s7
epoch=1, loss=0.6862
epoch=2, loss=0.3289
epoch=3, loss=0.2388
epoch=4, loss=0.1808
epoch=5, loss=0.1496
valid [full], NDCG@5: 0.6602, HR@5: 0.7985, NDCG@10: 0.7132, HR@10: 0.9634, MRR: 0.6406
valid [sampled], NDCG@5: 0.5497, HR@5: 0.5510, NDCG@10: 0.5556, HR@10: 0.5700, MRR: 0.5674
test [full], NDCG@5: 0.6524, HR@5: 0.8945, NDCG@10: 0.6835, HR@10: 0.9940, MRR: 0.5848
test [sampled], NDCG@5: 0.1813, HR@5: 0.1834, NDCG@10: 0.2007, HR@10: 0.2471, MRR: 0.2284
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-

## Rebuild result tables


In [18]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'use_time_attention_bias': config.get('use_time_attention_bias', False),
            'time_modeling_mode': config.get('time_modeling_mode'),
            'time_encoding': config.get('time_encoding'),
            'time_delta_column': config.get('time_delta_column'),
            'time_bucket_boundaries_parsed': config.get('time_bucket_boundaries_parsed'),
            'time_attention_bias_bucket_count': config.get('time_attention_bias_bucket_count'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


In [19]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

## NDCG@10 comparison summary


In [20]:
baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
timeaware_runs = [
    'attnbias_dstart_ml20_b9_s42',
    'attnbias_dstart_ml20_b9_s2024',
    'attnbias_dstart_ml20_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['time_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['time_variant'] = 'attention_bias_dstart_b9'

df_ndcg10 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'time_variant', 'use_time_embedding', 'use_time_attention_bias', 'time_modeling_mode',
    'time_delta_column', 'time_bucket_boundaries_parsed', 'time_attention_bias_bucket_count',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,use_time_embedding,use_time_attention_bias,time_modeling_mode,time_delta_column,time_bucket_boundaries_parsed,time_attention_bias_bucket_count,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,attnbias_dstart_ml20_b9_s7,7,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.731681,0.933234,0.694816,0.812322,0.674079,0.833933,1.000000,0.833460,0.998642,0.776636,0.584594,0.673558,0.550907,0.567329,0.571303,0.408188,0.553189,0.349894,0.366486,0.395330
1,attnbias_dstart_ml20_b9_s42,42,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.731547,0.976377,0.692852,0.854887,0.657784,0.857427,1.000000,0.844950,0.958941,0.810092,0.553607,0.600217,0.535631,0.543183,0.557929,0.535711,0.621374,0.504964,0.524491,0.530831
2,attnbias_dstart_ml20_b9_s2024,2024,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.727796,0.977380,0.682637,0.839902,0.653203,0.815191,1.000000,0.800095,0.953875,0.753665,0.551441,0.581497,0.539500,0.543537,0.560101,0.369433,0.517223,0.311595,0.332384,0.355765
3,anchor_ml20_s7,7,baseline,False,False,None,delta_prev_seconds,None,None,0.753294,0.961857,0.729167,0.883942,0.691225,0.913637,1.000000,0.903861,0.971918,0.885501,0.601240,0.661113,0.580044,0.594609,0.597937,0.480200,0.667438,0.420861,0.484209,0.443705
4,anchor_ml20_s42,42,baseline,False,False,None,delta_prev_seconds,None,None,0.750367,0.977546,0.708784,0.848505,0.681799,0.915296,1.000000,0.913814,0.995781,0.886842,0.583692,0.654340,0.556357,0.567822,0.578433,0.464891,0.652470,0.406641,0.473262,0.430220
5,anchor_ml20_s2024,2024,baseline,False,False,None,delta_prev_seconds,None,None,0.736821,0.967429,0.700301,0.850202,0.668534,0.840611,0.999865,0.819421,0.937829,0.789072,0.574314,0.616064,0.557801,0.563197,0.578608,0.323103,0.518524,0.261120,0.326822,0.292464


In [21]:
summary_ndcg10 = df_ndcg10.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


best_valid_full_ndcg@10           best_test_full_ndcg@10           best_valid_full_ndcg@5           best_test_full_ndcg@5           best_valid_full_mrr           best_test_full_mrr           best_valid_sampled_ndcg@10           best_test_sampled_ndcg@10           best_valid_sampled_ndcg@5           best_test_sampled_ndcg@5           best_valid_sampled_mrr           best_test_sampled_mrr          
                                            mean       std                   mean       std                   mean       std                  mean       std                mean       std               mean       std                       mean       std                      mean       std                      mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                            
attention_bias_dstart_b9                0.730341  0.002206               0.835517  0.021162               0.690102  0.006539              0.826168  0.023299            0.661689  0.010972           0.780131  0.028375                   0.563214  0.018547                  0.437777  0.086999                  0.542013  0.007942                 0.388818  0.102392               0.563111  0.007177              0.427309  0.091809
baseline                                0.746827  0.008788               0.889848  0.042649               0.712751  0.014836              0.879032  0.051864            0.680520  0.011400           0.853805  0.056064                   0.586415  0.013668                  0.422731  0.086620                  0.564734  0.013279                 0.362874  0.088408               0.584993  0.011211              0.388797  0.083699

Interpretation guide for NDCG@10:
- compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` first
- then check whether sampled and MRR move in the same direction
- baseline is reused; only attention-bias runs are newly trained here


## NDCG@5 comparison summary


In [22]:
baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
timeaware_runs = [
    'attnbias_dstart_ml20_b9_s42',
    'attnbias_dstart_ml20_b9_s2024',
    'attnbias_dstart_ml20_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['time_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['time_variant'] = 'attention_bias_dstart_b9'

df_ndcg5 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'time_variant', 'use_time_embedding', 'use_time_attention_bias', 'time_modeling_mode',
    'time_delta_column', 'time_bucket_boundaries_parsed', 'time_attention_bias_bucket_count',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,use_time_embedding,use_time_attention_bias,time_modeling_mode,time_delta_column,time_bucket_boundaries_parsed,time_attention_bias_bucket_count,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,attnbias_dstart_ml20_b9_s7,7,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.731681,0.933234,0.694816,0.812322,0.674079,0.833933,1.000000,0.833460,0.998642,0.776636,0.584594,0.673558,0.550907,0.567329,0.571303,0.408188,0.553189,0.349894,0.366486,0.395330
1,attnbias_dstart_ml20_b9_s42,42,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.728295,0.942107,0.695528,0.834303,0.667054,0.903459,1.000000,0.902459,0.997142,0.870589,0.570725,0.624454,0.549478,0.557178,0.570069,0.639984,0.700095,0.619380,0.635052,0.640352
2,attnbias_dstart_ml20_b9_s2024,2024,attention_bias_dstart_b9,False,True,attention_bias,delta_start_seconds,"[60.0, 600.0, 3600.0, 86400.0, 604800.0]",7,0.727796,0.977380,0.682637,0.839902,0.653203,0.815191,1.000000,0.800095,0.953875,0.753665,0.551441,0.581497,0.539500,0.543537,0.560101,0.369433,0.517223,0.311595,0.332384,0.355765
3,anchor_ml20_s7,7,baseline,False,False,None,delta_prev_seconds,None,None,0.753294,0.961857,0.729167,0.883942,0.691225,0.913637,1.000000,0.903861,0.971918,0.885501,0.601240,0.661113,0.580044,0.594609,0.597937,0.480200,0.667438,0.420861,0.484209,0.443705
4,anchor_ml20_s42,42,baseline,False,False,None,delta_prev_seconds,None,None,0.750367,0.977546,0.708784,0.848505,0.681799,0.915296,1.000000,0.913814,0.995781,0.886842,0.583692,0.654340,0.556357,0.567822,0.578433,0.464891,0.652470,0.406641,0.473262,0.430220
5,anchor_ml20_s2024,2024,baseline,False,False,None,delta_prev_seconds,None,None,0.736821,0.967429,0.700301,0.850202,0.668534,0.840611,0.999865,0.819421,0.937829,0.789072,0.574314,0.616064,0.557801,0.563197,0.578608,0.323103,0.518524,0.261120,0.326822,0.292464


In [23]:
summary_ndcg5 = df_ndcg5.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                            mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                
attention_bias_dstart_b9                0.729257  0.002114              0.950907  0.023352               0.690994  0.007246             0.828842  0.014579            0.664779  0.010623               0.850861  0.046505             1.000000  0.000000              0.845338  0.052206             0.98322  0.025424           0.800297  0.061949                   0.568920  0.016650                 0.626503  0.046065                  0.546628  0.006215                0.556015  0.011938               0.567158  0.006142                  0.472535  0.146304                0.590169  0.096883                 0.426956  0.167740               0.444641  0.165781              0.463816  0.154159
baseline                                0.746827  0.008788              0.968944  0.007954               0.712751  0.014836             0.860883  0.019988            0.680520  0.011400               0.889848  0.042649             0.999955  0.000078              0.879032  0.051864             0.96851  0.029126           0.853805  0.056064                   0.586415  0.013668                 0.643839  0.024291                  0.564734  0.013279                0.575209  0.016959               0.584993  0.011211                  0.422731  0.086620                0.612810  0.081997                 0.362874  0.088408               0.428098  0.087878              0.388797  0.083699

Interpretation guide for NDCG@5:
- compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` first
- then check whether sampled and MRR move in the same direction
- baseline is reused; only attention-bias runs are newly trained here
